# GUDS-EDL: New Baselines and Loss-Matched Sparse Sweeps (Kaggle Edition)

This notebook provides a fully automated pipeline to train and evaluate the new revision baselines on Kaggle:
1. **Class-Balanced EDL** (with class pooling loss and a learnable Dirichlet prior $\bm{\beta}$)
2. **Dense EDL + EFL** (loss control for dense networks)
3. **Static 2:4 EDL + EFL** (loss control for fixed sparse networks)
4. **RigL-style 2:4 + EFL** (loss control for dynamic sparse networks)

### Optimization Strategy for Maximum Performance (Strict Protocol):
- **Checkpoint Selection (`--checkpoint_selection pauc`):** Saves the model state at the best validation `pAUC 0.80` rather than the final epoch. This prevents overfitting on majority classes and captures the model at its peak ranking capability.
- **Extended Training Cycle:** 40 epochs of training with `--checkpoint_start_epoch 10` and evaluating validation metrics every 2 epochs (`--checkpoint_eval_every 2`) to ensure fine-grained selection.
- **NVIDIA Hardware Layout Parity (`--protocol_profile nvidia_v3`):** Uses the hardware-compatible layout for 2:4 sparsity, aligning perfectly with TensorRT deployment claims.

In [ ]:
# Cell 1: Pull repository changes from GitHub
from pathlib import Path
import subprocess
import os
import sys

repo_url = "https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git"
working_dir = Path("/kaggle/working")
repo_dir = working_dir / "MDEP-Microglial-Driven-Evidential-Pruning"

if not repo_dir.exists():
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    print("Repository already exists. Pulling latest updates...")
    subprocess.run(["git", "pull"], cwd=str(repo_dir), check=True)

# Change directory to the cloned repo
os.chdir(str(repo_dir))
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print(f"Working directory switched to: {os.getcwd()}")

In [ ]:
# Cell 2: Verify and Setup Datasets
import os
from pathlib import Path

# Candidates for dataset root directories
dataset_candidates = [
    Path("/kaggle/input/isic-2024-challenge"),
    Path("/kaggle/input/competitions/isic-2024-challenge")
]

isic_root = None
for candidate in dataset_candidates:
    if candidate.exists() and (candidate / "train-metadata.csv").exists():
        isic_root = candidate
        break

if isic_root:
    print(f"Found ISIC dataset at: {isic_root}")
    os.environ["ISIC_ROOT"] = str(isic_root)
else:
    raise FileNotFoundError(
        "ISIC dataset train-metadata.csv not found. "
        "Please make sure to mount the 'isic-2024-challenge' dataset to your Kaggle session."
    )

# Pre-requisite package install
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "matplotlib", "pandas", "h5py", "tqdm", "scipy", "tabulate"], check=True)
print("Dependencies installed successfully.")

In [ ]:
# Cell 3: Execute baseline runs with optimal protocol selection
import shlex

experiments = [
    "dense_edl_efl",
    "static_24_edl_efl",
    "rigl_style_24_efl",
    "class_balanced_edl"
]
seeds = ["42", "123", "456"]

for exp in experiments:
    print("\n" + "=" * 96)
    print(f"RUNNING EXPERIMENT: {exp} | Seeds: {seeds}")
    print("=" * 96)
    
    cmd = [
        sys.executable, "-u", "experiments/isic_paper_experiments.py",
        "--experiment", exp,
        "--epochs", "40",                  # Standard protocol epochs
        "--batch_size", "32",
        "--lr", "4e-5",
        "--seeds", *seeds,
        "--split_seed", "42",
        "--checkpoint_selection", "pauc",   # Extract validation-best model based on pAUC
        "--checkpoint_eval_every", "2",
        "--checkpoint_start_epoch", "10",
        "--protocol_profile", "nvidia_v3"
    ]
    
    print(f"$ {shlex.join(cmd)}", flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    
    if process.returncode != 0:
        print(f"[ERROR] {exp} failed with code {process.returncode}")
    else:
        print(f"[DONE] Successfully completed {exp}")

In [ ]:
# Cell 4: Parse outputs and print aggregated LaTeX tables
import json
import pandas as pd
import numpy as np

output_dir = Path("/kaggle/working/paper_experiment_outputs/isic")
experiments_list = [
    ("class_balanced_edl", "Class-Balanced EDL"),
    ("dense_edl_efl", "Dense EDL + EFL"),
    ("static_24_edl_efl", "Static 2:4 + EFL"),
    ("rigl_style_24_efl", "RigL-style 2:4 + EFL")
]
confirm_seeds = [42, 123, 456]

rows = []
for exp_name, label in experiments_list:
    pauc_list = []
    auroc_list = []
    pr_list = []
    ece_list = []
    acc_list = []
    
    for seed in confirm_seeds:
        p = output_dir / exp_name / f"seed_{seed}" / "metrics.json"
        if p.exists():
            data = json.loads(p.read_text(encoding='utf-8'))
            m = data.get("metrics", {})
            pauc_list.append(m.get("pauc", 0.0))
            auroc_list.append(m.get("macro_auroc", 0.0))
            pr_list.append(m.get("pr_auc", 0.0))
            ece_list.append(m.get("ece_adaptive", 0.0))
            acc_list.append(m.get("balanced_accuracy_default", m.get("balanced_accuracy", 0.0)))
            
    if pauc_list:
        rows.append({
            "Model": label,
            "pAUC": f"{np.mean(pauc_list):.4f} \u00b1 {np.std(pauc_list):.4f}",
            "Macro AUROC": f"{np.mean(auroc_list):.4f} \u00b1 {np.std(auroc_list):.4f}",
            "PR AUC": f"{np.mean(pr_list):.4f} \u00b1 {np.std(pr_list):.4f}",
            "ECE": f"{np.mean(ece_list):.4f} \u00b1 {np.std(ece_list):.4f}",
            "Bal. Acc": f"{np.mean(acc_list):.4f} \u00b1 {np.std(acc_list):.4f}"
        })

df_results = pd.DataFrame(rows)
print("\n========================================================================")
print("AGGREGATED METRICS SUMMARY (Markdown)")
print("========================================================================")
print(df_results.to_markdown(index=False))

print("\n========================================================================")
print("LATEX TABLE CODE FOR PAPER")
print("========================================================================")
for _, row in df_results.iterrows():
    print(f"{row['Model']} & {row['Bal. Acc']} & {row['pAUC']} & {row['Macro AUROC']} & {row['ECE']} \\\\")